# GainSplit · 01 · Data preprocessing

This notebook turns the **raw** datasets into the exact files every experiment will use. Run it **once**. All experiments then read its output, so every model sees the same data.

**What it produces** (in `/kaggle/working/processed/`):

| File | Contents |
|---|---|
| `headlines_train.csv`, `headlines_dev.csv`, `headlines_test.csv` | `id, text, label` (80 / 10 / 10 split) |
| `budgets.json` | which train `id`s keep their label, for every seed × label budget |
| `helper_labels.csv` | sentiment and emotion helper labels (class + probabilities) for every headline |
| `helper_emotion28.csv` | raw scores for all 28 GoEmotions emotions (kept so a different mapping can be tried later, RQ2) |
| `isarcasm_train.csv`, `isarcasm_test.csv` | `id, text, label` for the transfer test (RQ3) |
| `manifest.json` | settings, row counts, removed rows, library versions, file hashes. **Numbers for the paper's Data section.** |

**Steps**
1. Setup and settings
2. Find and verify the raw files
3. Headlines: load → drop the leaking link → clean text → remove empty rows and duplicates → check lengths → split → label budgets
4. Helper labels from two pretrained "teacher" models (needs GPU)
5. iSarcasmEval: load → clean the same way → check
6. Save everything and write the manifest

### Before you press "Run All"
1. **Settings → Accelerator → GPU** (T4 or P100). Step 4 runs two models over ~28k headlines: a few minutes on GPU, much longer on CPU.
2. **Settings → Internet → On.** The notebook downloads the BERT tokenizer and the two teacher models from Hugging Face.
3. **Add Input → your raw-data dataset.** It must contain the three unedited files: `Sarcasm_Headlines_Dataset.json`, `train.En.csv` (iSarcasmEval `train/`) and `task_A_En_test.csv` (iSarcasmEval `test/`). Folder names do not matter; the notebook searches for the file names.

## 1 · Setup

Imports and a quick check of the hardware. If the last line says `cuda: False`, stop and switch the accelerator to GPU.

In [ ]:
import json, re, html, hashlib, unicodedata, platform
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers
import sklearn
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification

pd.set_option("display.max_colwidth", 120)
print("python", platform.python_version(), "| pandas", pd.__version__, "| torch", torch.__version__,
      "| transformers", transformers.__version__, "| sklearn", sklearn.__version__)
print("cuda:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

### Settings

Everything you might want to change is here; nothing below needs editing.

- **`KEEP_PUNCTUATION`**: `False` = punctuation removed, following the seed paper. It also removes one outlet-style shortcut (The Onion vs HuffPost editing style), and prior studies report small, task-dependent effects of this step on transformers (see `vault/Literature/Punctuation Evidence.md`). The teacher models read the same text, so helper labels are made from punctuation-free text too. Setting `True` is the optional robustness check.
- **`DROP_DUPLICATES`**: the same headline in train *and* test lets the model be tested on something it has already seen. With `True`, one copy of each headline is kept, and headlines that appear with *both* labels are removed entirely.
- **`SPLIT_SEED`** fixes the one train/dev/test split. It is separate from the training **`SEEDS`**, which only decide *which* train headlines keep their label.
- **`BUDGETS`**: % of train that keeps its label. The rest becomes the unlabeled pool **U**.

In [ ]:
KEEP_PUNCTUATION = False
DROP_DUPLICATES  = True

SPLIT_SEED = 42
SPLIT      = (0.8, 0.1, 0.1)          # train / dev / test
SEEDS      = [0, 1, 2, 3, 4]           # Pass 1 uses the first three
BUDGETS    = [1, 5, 10, 30, 100]       # % of train that keeps its label

TOKENIZER_NAME  = "bert-base-uncased"  # the backbone used by every arm
MAX_LEN         = 48                   # tokens BERT reads per headline (incl. [CLS] and [SEP])
SENTIMENT_MODEL = "cardiffnlp/twitter-roberta-base-sentiment"
EMOTION_MODEL   = "SamLowe/roberta-base-go_emotions"
TEACHER_BATCH   = 128

# Known facts about the raw files, used as checks (see vault/Datasets/)
EXPECTED_HEADLINES_ROWS_SARC = (28619, 13634)
EXPECTED_HEADLINES_SHA256 = "ab0faf86c2dbbef47a95080bc7601981f3ea9dc48ccca73382f90b9a56e02b49"
EXPECTED_ISARC = {"train": (3468, 867), "test": (1400, 200)}

INPUT_ROOT = Path("/kaggle/input")
OUT_DIR    = Path("/kaggle/working/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

report = {"settings": dict(
    KEEP_PUNCTUATION=KEEP_PUNCTUATION, DROP_DUPLICATES=DROP_DUPLICATES, SPLIT_SEED=SPLIT_SEED,
    SPLIT=SPLIT, SEEDS=SEEDS, BUDGETS=BUDGETS, TOKENIZER_NAME=TOKENIZER_NAME, MAX_LEN=MAX_LEN,
    SENTIMENT_MODEL=SENTIMENT_MODEL, EMOTION_MODEL=EMOTION_MODEL)}

## 2 · Find and verify the raw files

The notebook searches `/kaggle/input` for the three file names, so it works whatever you named your Kaggle dataset.

It then computes a **SHA-256 fingerprint** of each file. If even one character changed, the fingerprint changes. The Headlines fingerprint is compared with the one recorded when the file was first downloaded, which shows the input is the untouched original.

In [ ]:
def find_one(name):
    hits = sorted(INPUT_ROOT.rglob(name))
    if not hits:
        raise FileNotFoundError(f"{name} not found under {INPUT_ROOT}. Use 'Add Input' to attach the raw dataset.")
    if len(hits) > 1:
        print(f"Note: {len(hits)} copies of {name} found; using {hits[0]}")
    return hits[0]

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

HEADLINES_PATH   = find_one("Sarcasm_Headlines_Dataset.json")
ISARC_TRAIN_PATH = find_one("train.En.csv")
ISARC_TEST_PATH  = find_one("task_A_En_test.csv")

report["inputs"] = {}
for p in [HEADLINES_PATH, ISARC_TRAIN_PATH, ISARC_TEST_PATH]:
    report["inputs"][p.name] = {"path": str(p), "sha256": sha256(p)}
    print(f"{p.name:35s} {report['inputs'][p.name]['sha256']}")

if report["inputs"][HEADLINES_PATH.name]["sha256"] == EXPECTED_HEADLINES_SHA256:
    print("\nHeadlines file matches the recorded original. ✔")
else:
    print("\nWARNING: Headlines file differs from the recorded original. Check you uploaded the unedited file.")

## 3 · Headlines

### 3.1 Load and check

The file is **JSON Lines**: one headline per line with `is_sarcastic`, `headline` and `article_link`.
Expected: **28,619 rows, 13,634 sarcastic (47.6%)**. The seed paper reports different, inconsistent counts, so the paper should state these numbers.

In [ ]:
raw = pd.read_json(HEADLINES_PATH, lines=True)
n_rows, n_sarc = len(raw), int(raw["is_sarcastic"].sum())
print("columns:", raw.columns.tolist())
print(f"rows: {n_rows:,}  sarcastic: {n_sarc:,} ({n_sarc / n_rows:.1%})")
assert set(raw["is_sarcastic"].unique()) <= {0, 1}, "labels must be 0/1"
if (n_rows, n_sarc) != EXPECTED_HEADLINES_ROWS_SARC:
    print("WARNING: counts differ from the expected", EXPECTED_HEADLINES_ROWS_SARC)
report["headlines_raw"] = {"rows": n_rows, "sarcastic": n_sarc}
raw.head()

### 3.2 Drop `article_link` and give every headline an `id`

`article_link` contains `theonion.com` (sarcastic) or `huffingtonpost.com` (not sarcastic), so it **is** the label. Keeping it would be cheating.

`id` = the row number in the raw file. It never changes, so any prediction can be traced back to the original headline.

In [ ]:
print("link domain vs label (why the link must go):")
print(pd.crosstab(raw["article_link"].str.extract(r"https?://(?:www\.)?([^/]+)")[0], raw["is_sarcastic"]))

hl = raw.drop(columns=["article_link"]).rename(columns={"headline": "raw_text", "is_sarcastic": "label"})
hl.insert(0, "id", np.arange(len(hl)))
hl.head(3)

### 3.3 Clean the text (lightly)

BERT was trained on normal sentences, so we change as little as possible:
1. Decode HTML codes (`&amp;` → `&`).
2. Unicode normalisation (NFKC) and curly quotes → straight quotes, so the same character is always written the same way.
3. Optionally remove punctuation (`KEEP_PUNCTUATION = False`).
4. Collapse repeated spaces and trim the ends.

We do **not** lowercase (the `bert-base-uncased` tokenizer does that itself), remove stop-words or stem. Those steps help old bag-of-words models but remove information BERT uses.

The same `clean_text` function is used for iSarcasmEval later, so both datasets are treated identically.

In [ ]:
QUOTES = str.maketrans({"‘": "'", "’": "'", "“": '"', "”": '"'})
PUNCT  = re.compile(r"[^\w\s]")
SPACES = re.compile(r"\s+")

def clean_text(t):
    if not isinstance(t, str):
        return ""
    t = html.unescape(t)
    t = unicodedata.normalize("NFKC", t).translate(QUOTES)
    if not KEEP_PUNCTUATION:
        t = PUNCT.sub(" ", t)
    return SPACES.sub(" ", t).strip()

hl["text"] = hl["raw_text"].map(clean_text)
changed = hl[hl["text"] != hl["raw_text"]]
print(f"headlines changed by cleaning: {len(changed):,}")
report["headlines_changed_by_cleaning"] = int(len(changed))
changed.sample(min(10, len(changed)), random_state=0)[["raw_text", "text"]] if len(changed) else None

### 3.4 Remove empty headlines and duplicates

- **Empty:** nothing to learn from.
- **Duplicates:** detected on the cleaned, lower-cased text (the uncased model cannot tell `Trump` from `trump` anyway).
  - Same text, same label → keep the first copy.
  - Same text, **different** labels → remove all copies; there is no correct answer to learn.

Write the numbers printed here into the paper.

In [ ]:
empty = hl["text"].str.len() == 0
print(f"empty headlines: {int(empty.sum())}")
hl = hl[~empty].copy()

hl["_key"] = hl["text"].str.lower()
groups = hl.groupby("_key")["label"].agg(["size", "nunique"])
conflict_keys = set(groups.index[groups["nunique"] > 1])
n_extra_copies = int((groups["size"] - 1).clip(lower=0).sum())
n_conflict_rows = int(hl["_key"].isin(conflict_keys).sum())
print(f"texts that appear more than once: {int((groups['size'] > 1).sum()):,}")
print(f"extra copies (rows beyond the first): {n_extra_copies:,}")
print(f"rows whose text appears with both labels: {n_conflict_rows:,}")

if DROP_DUPLICATES:
    hl = hl[~hl["_key"].isin(conflict_keys)]
    hl = hl.drop_duplicates(subset="_key", keep="first")
hl = hl.drop(columns=["_key", "raw_text"]).reset_index(drop=True)

print(f"\nheadlines kept: {len(hl):,}  sarcastic: {int(hl['label'].sum()):,} ({hl['label'].mean():.1%})")
report["headlines_cleaning"] = {"empty_removed": int(empty.sum()), "duplicate_extra_copies": n_extra_copies,
                                "conflicting_label_rows": n_conflict_rows, "rows_kept": int(len(hl)),
                                "sarcastic_kept": int(hl["label"].sum())}

### 3.5 Check headline lengths

BERT reads at most `MAX_LEN` tokens (word pieces) per headline; anything longer is cut off. This cell confirms that 48 is enough. Expect almost no headline above it.

In [ ]:
bert_tok = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
lengths = np.array([len(ids) for ids in bert_tok(hl["text"].tolist())["input_ids"]])
pct = {p: int(np.percentile(lengths, p)) for p in (50, 90, 99, 100)}
n_over = int((lengths > MAX_LEN).sum())
print("token length percentiles (incl. [CLS]/[SEP]):", pct)
print(f"headlines longer than {MAX_LEN} tokens: {n_over} ({n_over / len(hl):.2%})")
report["headlines_token_lengths"] = {"percentiles": pct, "over_max_len": n_over}

### 3.6 Split once: 80% train / 10% dev / 10% test

- **Stratified:** each part keeps the same sarcastic / non-sarcastic ratio.
- **Train** is where label budgets are taken from. **Dev** is for choosing settings (hyperparameters). **Test** is used only for the final numbers. Never tune on it.
- The seed paper used 80/20 with no dev set. We add a dev set so that settings are never chosen on the test set; state this in the paper.

The checks at the end make sure no headline is in two parts.

In [ ]:
train_df, rest = train_test_split(hl, test_size=SPLIT[1] + SPLIT[2], stratify=hl["label"], random_state=SPLIT_SEED)
dev_df, test_df = train_test_split(rest, test_size=SPLIT[2] / (SPLIT[1] + SPLIT[2]), stratify=rest["label"], random_state=SPLIT_SEED)
train_df, dev_df, test_df = (d.sort_values("id").reset_index(drop=True) for d in (train_df, dev_df, test_df))

parts = {"train": train_df, "dev": dev_df, "test": test_df}
for a in parts:
    for b in parts:
        if a < b:
            assert not set(parts[a]["id"]) & set(parts[b]["id"]), f"id overlap {a}/{b}"
            assert not set(parts[a]["text"].str.lower()) & set(parts[b]["text"].str.lower()) or not DROP_DUPLICATES, f"text overlap {a}/{b}"

split_table = pd.DataFrame({k: {"rows": len(d), "sarcastic": int(d["label"].sum()), "sarcastic_share": round(d["label"].mean(), 4)}
                            for k, d in parts.items()}).T
report["headlines_split"] = split_table.to_dict(orient="index")
split_table

### 3.7 Label budgets

For each training seed and each budget *p*, choose which train headlines **keep** their label (the labeled set **L**). The others form the unlabeled pool **U**: same text, label hidden.

How the choice is made:
- **Stratified:** L keeps the sarcastic ratio of train.
- **Nested:** for one seed, the 1% set is inside the 5% set, which is inside the 10% set, and so on. Going from 5% to 10% only *adds* labels, so differences between budgets come from the amount of labels, not from a different random draw.
- Saved as lists of `id`s. Every arm reads the same lists, so all arms see exactly the same labeled headlines.

U is not saved separately: U = train ids not in L.

In [ ]:
def nested_budgets(train, seed, budgets):
    rng = np.random.default_rng(seed)
    order = {c: rng.permutation(g["id"].to_numpy()) for c, g in train.groupby("label")}
    out = {}
    for p in sorted(budgets):
        ids = []
        for c, perm in order.items():
            k = max(1, int(round(len(perm) * p / 100)))
            ids.extend(int(i) for i in perm[:k])
        out[str(p)] = sorted(ids)
    return out

budgets = {str(s): nested_budgets(train_df, s, BUDGETS) for s in SEEDS}

# checks: nested, inside train, 100% = all of train
train_ids = set(train_df["id"])
for s, by_p in budgets.items():
    ps = sorted(by_p, key=int)
    for small, big in zip(ps, ps[1:]):
        assert set(by_p[small]) <= set(by_p[big]), f"seed {s}: {small}% not inside {big}%"
    assert set(by_p[ps[-1]]) <= train_ids
    if "100" in by_p:
        assert set(by_p["100"]) == train_ids

label_of = dict(zip(train_df["id"], train_df["label"]))
budget_table = pd.DataFrame({f"seed {s}": {f"{p}%": f"{len(ids):,} (sarc {np.mean([label_of[i] for i in ids]):.1%})"
                                         for p, ids in by_p.items()} for s, by_p in budgets.items()})
report["budget_sizes"] = {s: {p: len(ids) for p, ids in by_p.items()} for s, by_p in budgets.items()}
budget_table

## 4 · Helper labels from teacher models

The helper heads need sentiment and emotion labels, which Headlines does not have. Two public pretrained models (**teachers**) produce them for **every** headline: train (labeled and unlabeled), dev and test. Only train labels are used for training. Dev/test labels are kept for analysis, e.g. checking whether helper labels simply track the outlet (RQ3).

**Sentiment:** `cardiffnlp/twitter-roberta-base-sentiment` → negative / neutral / positive (softmax probabilities; class order 0 = negative, 1 = neutral, 2 = positive, as stated on the model card). Unlike the seed paper, neutral is **kept**: dropping it would remove headlines from the data.

**Emotion:** `SamLowe/roberta-base-go_emotions` scores 28 emotions independently (sigmoid; scores do not sum to 1). We group them into 5 classes. The grouping follows the GoEmotions authors' Ekman grouping, with *disgust* merged into *anger* and *surprise* merged into *neutral_other*. This mapping is **our choice** (the seed paper's is unknown) and must be printed in the paper:

| Class | GoEmotions labels |
|---|---|
| joy | admiration, amusement, approval, caring, desire, excitement, gratitude, joy, love, optimism, pride, relief |
| sadness | disappointment, embarrassment, grief, remorse, sadness |
| anger | anger, annoyance, disapproval, disgust |
| fear | fear, nervousness |
| neutral_other | neutral, confusion, curiosity, realization, surprise |

A class's score = the **highest** score among its labels. "Highest" rather than "sum" so that *joy*, which has 12 labels, does not win just by having more of them. The five scores are then rescaled to sum to 1, giving a probability for each class. The hard label is the class with the largest probability. Both are saved: RQ2 compares hard and soft labels.

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

@torch.no_grad()
def teacher_scores(model_name, texts, activation):
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to(DEVICE).eval()
    chunks = []
    for i in range(0, len(texts), TEACHER_BATCH):
        enc = tok(texts[i:i + TEACHER_BATCH], padding=True, truncation=True, max_length=128, return_tensors="pt").to(DEVICE)
        logits = model(**enc).logits.float()
        probs = torch.softmax(logits, dim=-1) if activation == "softmax" else torch.sigmoid(logits)
        chunks.append(probs.cpu().numpy())
    names = [model.config.id2label[i] for i in range(model.config.num_labels)]
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return np.concatenate(chunks), names

all_hl = pd.concat([d.assign(split=k) for k, d in parts.items()], ignore_index=True)
texts = all_hl["text"].tolist()
print(f"scoring {len(texts):,} headlines on {DEVICE}")

### 4.1 Sentiment

In [ ]:
SENT_CLASSES = ["negative", "neutral", "positive"]
sent_probs, sent_names = teacher_scores(SENTIMENT_MODEL, texts, "softmax")
assert [n.lower() for n in sent_names] in (["label_0", "label_1", "label_2"], SENT_CLASSES), sent_names

for j, c in enumerate(SENT_CLASSES):
    all_hl[f"sent_p_{c}"] = sent_probs[:, j].round(5)
all_hl["sent_label"] = np.array(SENT_CLASSES)[sent_probs.argmax(1)]
print(all_hl["sent_label"].value_counts())

**Spot check:** read the most confident examples of each class. If the "positive" headlines look negative, the class order is wrong. Stop and fix it before going on.

In [ ]:
for j, c in enumerate(SENT_CLASSES):
    top = all_hl.iloc[np.argsort(-sent_probs[:, j])[:5]]
    print(f"--- most {c} ---")
    for t in top["text"]:
        print("  ", t)

### 4.2 Emotion

In [ ]:
EMOTION_GROUPS = {
    "joy":           ["admiration", "amusement", "approval", "caring", "desire", "excitement", "gratitude",
                      "joy", "love", "optimism", "pride", "relief"],
    "sadness":       ["disappointment", "embarrassment", "grief", "remorse", "sadness"],
    "anger":         ["anger", "annoyance", "disapproval", "disgust"],
    "fear":          ["fear", "nervousness"],
    "neutral_other": ["neutral", "confusion", "curiosity", "realization", "surprise"],
}
EMO_CLASSES = list(EMOTION_GROUPS)

emo28, emo_names = teacher_scores(EMOTION_MODEL, texts, "sigmoid")
mapped = [e for es in EMOTION_GROUPS.values() for e in es]
assert sorted(mapped) == sorted(emo_names) and len(mapped) == 28, "mapping must cover the 28 labels exactly once"

grouped = np.stack([emo28[:, [emo_names.index(e) for e in EMOTION_GROUPS[c]]].max(1) for c in EMO_CLASSES], axis=1)
emo5 = grouped / grouped.sum(1, keepdims=True)

for j, c in enumerate(EMO_CLASSES):
    all_hl[f"emo_p_{c}"] = emo5[:, j].round(5)
all_hl["emo_label"] = np.array(EMO_CLASSES)[emo5.argmax(1)]
print(all_hl["emo_label"].value_counts())

### 4.3 First look: do helper labels relate to sarcasm? (train only)

Shares of sarcastic headlines within each helper class, **train only** (test labels are not looked at). Large differences mean the helper labels carry information about the label, which is what the helper heads are supposed to exploit. Whether that information is "sarcasm" or just "Onion style" is a question for RQ3.

In [ ]:
tr = all_hl[all_hl["split"] == "train"]
display(pd.crosstab(tr["sent_label"], tr["label"], normalize="index").round(3).rename(columns={0: "not sarc", 1: "sarc"}))
display(pd.crosstab(tr["emo_label"], tr["label"], normalize="index").round(3).rename(columns={0: "not sarc", 1: "sarc"}))

In [ ]:
helper_cols = ["id", "split", "sent_label"] + [f"sent_p_{c}" for c in SENT_CLASSES] + ["emo_label"] + [f"emo_p_{c}" for c in EMO_CLASSES]
helper_df = all_hl[helper_cols].sort_values("id").reset_index(drop=True)
emo28_df = pd.DataFrame(emo28.round(5), columns=emo_names).assign(id=all_hl["id"].to_numpy(), split=all_hl["split"].to_numpy())
emo28_df = emo28_df[["id", "split"] + emo_names].sort_values("id").reset_index(drop=True)
report["helper_label_counts"] = {"sentiment": helper_df["sent_label"].value_counts().to_dict(),
                                 "emotion": helper_df["emo_label"].value_counts().to_dict()}
report["emotion_mapping"] = EMOTION_GROUPS
helper_df.head()

## 5 · iSarcasmEval (transfer test, RQ3)

Tweets labelled sarcastic / not sarcastic **by their own authors**. Only two files are used:
- `train.En.csv`: 3,468 tweets (867 sarcastic). Columns: an unnamed row number, `tweet`, `sarcastic`, `rephrase`, and six sarcasm-type flags. Only `tweet` and `sarcastic` are kept.
- `task_A_En_test.csv`: 1,400 tweets (200 sarcastic), the official Task A test set.

Same `clean_text` as Headlines. Rows with a missing tweet are removed and counted. Duplicates are only **reported**, not removed, so the official test set stays comparable with other papers.

Main score for this dataset (set by its authors): **F1 of the sarcastic class** (`pos_label=1`), not accuracy.

In [ ]:
def load_isarc(path, text_col, name):
    df = pd.read_csv(path)
    print(f"{name}: columns {df.columns.tolist()}")
    df = pd.DataFrame({"id": np.arange(len(df)), "text": df[text_col].map(clean_text), "label": df["sarcastic"]})
    missing = df["text"].str.len() == 0
    df = df[~missing].reset_index(drop=True)
    assert set(df["label"].unique()) <= {0, 1}
    dups = int(df["text"].str.lower().duplicated().sum())
    exp = EXPECTED_ISARC[name]
    print(f"  rows {len(df) + int(missing.sum()):,} (expected {exp[0]:,}), sarcastic {int(df['label'].sum()):,} (expected {exp[1]:,})")
    print(f"  missing/empty text removed: {int(missing.sum())}, duplicate texts (kept): {dups}")
    report[f"isarcasm_{name}"] = {"rows_raw": int(len(df) + missing.sum()), "missing_removed": int(missing.sum()),
                                  "duplicates_kept": dups, "rows_kept": int(len(df)), "sarcastic_kept": int(df["label"].sum())}
    return df

isarc_train = load_isarc(ISARC_TRAIN_PATH, "tweet", "train")
isarc_test  = load_isarc(ISARC_TEST_PATH, "text", "test")

overlap = set(isarc_train["text"].str.lower()) & set(isarc_test["text"].str.lower())
print(f"\ntweets in both iSarcasmEval train and test: {len(overlap)}")
report["isarcasm_train_test_overlap"] = len(overlap)
isarc_test.head()

## 6 · Save everything

Writes the files to `/kaggle/working/processed/` and a `manifest.json` with all settings and counts, plus a fingerprint of every output file. Experiment notebooks can record that fingerprint, which proves which data a result was made from.

In [ ]:
COLS = ["id", "text", "label"]
outputs = {
    "headlines_train.csv": train_df[COLS], "headlines_dev.csv": dev_df[COLS], "headlines_test.csv": test_df[COLS],
    "helper_labels.csv": helper_df, "helper_emotion28.csv": emo28_df,
    "isarcasm_train.csv": isarc_train[COLS], "isarcasm_test.csv": isarc_test[COLS],
}
for name, df in outputs.items():
    df.to_csv(OUT_DIR / name, index=False)
with open(OUT_DIR / "budgets.json", "w") as f:
    json.dump(budgets, f)

report["versions"] = {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
                      "torch": torch.__version__, "transformers": transformers.__version__, "sklearn": sklearn.__version__}
report["outputs"] = {p.name: {"bytes": p.stat().st_size, "sha256": sha256(p)} for p in sorted(OUT_DIR.iterdir()) if p.name != "manifest.json"}
with open(OUT_DIR / "manifest.json", "w") as f:
    json.dump(report, f, indent=2, default=str)

for p in sorted(OUT_DIR.iterdir()):
    print(f"{p.name:25s} {p.stat().st_size / 1e6:6.2f} MB")

## Summary for the paper

The cell below prints the numbers that go into the **Data** section. Copy them into the vault (`vault/Datasets/Headlines Dataset.md` and `vault/Datasets/iSarcasmEval.md`).

In [ ]:
c = report["headlines_cleaning"]
n_raw = report["headlines_raw"]["rows"]
print(f"Headlines: {n_raw:,} raw rows; kept {c['rows_kept']:,} ({c['sarcastic_kept']:,} sarcastic); removed {n_raw - c['rows_kept']:,} in total.")
print(f"  empty: {c['empty_removed']}; extra duplicate copies found: {c['duplicate_extra_copies']}; "
      f"rows whose text had both labels: {c['conflicting_label_rows']}; duplicates dropped: {DROP_DUPLICATES}")
print("Split:", {k: v["rows"] for k, v in report["headlines_split"].items()})
print(f"Punctuation kept: {KEEP_PUNCTUATION}. Headlines over {MAX_LEN} tokens: {report['headlines_token_lengths']['over_max_len']}.")
print("iSarcasmEval train/test kept:", report["isarcasm_train"]["rows_kept"], "/", report["isarcasm_test"]["rows_kept"])

## Next: use the output in experiment notebooks

1. Click **Save Version → Save & Run All (Commit)**. Kaggle runs the notebook from top to bottom and stores `/kaggle/working/processed/` as this notebook's **output**.
2. In any experiment notebook: **Add Input → Notebooks → this notebook**. The files appear under `/kaggle/input/<this-notebook-name>/processed/`.
3. Do not re-run this notebook after experiments have started. If you must (for example, you change `KEEP_PUNCTUATION`), every experiment has to be re-run on the new output.